In [1]:
%%writefile reduction.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define N 1024
#define ITERS 1000

// ---------------- GPU KERNEL ----------------
__global__ void reduceSum(float *input, float *output) {
    __shared__ float shared[1024];

    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;

    // Load into shared memory
    shared[tid] = input[i];
    __syncthreads();

    // Reduction
    for (int stride = blockDim.x / 2; stride > 0; stride /= 2) {
        if (tid < stride) {
            shared[tid] += shared[tid + stride];
        }
        __syncthreads();
    }

    // Store result
    if (tid == 0) {
        output[blockIdx.x] = shared[0];
    }
}

// ---------------- MAIN ----------------
int main() {
    int size = N * sizeof(float);

    float *h_input = (float*)malloc(size);
    float *h_output = (float*)malloc(sizeof(float));

    // Initialize array
    for (int i = 0; i < N; i++) {
        h_input[i] = 1.0f;
    }

    float *d_input, *d_output;
    cudaMalloc((void**)&d_input, size);
    cudaMalloc((void**)&d_output, sizeof(float));

    cudaMemcpy(d_input, h_input, size, cudaMemcpyHostToDevice);

    // -------- GPU REDUCTION TIMING --------
    cudaEvent_t gpuStart, gpuStop;
    cudaEventCreate(&gpuStart);
    cudaEventCreate(&gpuStop);

    cudaEventRecord(gpuStart);
    for (int iter = 0; iter < ITERS; iter++) {
        reduceSum<<<1, N>>>(d_input, d_output);
    }
    cudaMemcpy(h_output, d_output, sizeof(float), cudaMemcpyDeviceToHost);
    cudaEventRecord(gpuStop);
    cudaEventSynchronize(gpuStop);

    float gpu_time_ms = 0.0f;
    cudaEventElapsedTime(&gpu_time_ms, gpuStart, gpuStop);
    gpu_time_ms /= ITERS;

    // -------- CPU REDUCTION TIMING --------
    float cpu_sum = 0.0f;
    clock_t cpuStart = clock();
    for (int iter = 0; iter < ITERS; iter++) {
        cpu_sum = 0.0f;
        for (int i = 0; i < N; i++) {
            cpu_sum += h_input[i];
        }
    }
    clock_t cpuStop = clock();

    double cpu_time_ms = (1000.0 * (double)(cpuStop - cpuStart) / CLOCKS_PER_SEC) / ITERS;

    printf("GPU Sum: %f\n", h_output[0]);
    printf("CPU Sum: %f\n", cpu_sum);
    printf("Average GPU time per reduction: %.6f ms\n", gpu_time_ms);
    printf("Average CPU time per reduction: %.6f ms\n", cpu_time_ms);
    if (gpu_time_ms > 0.0f) {
        printf("Speedup (CPU/GPU): %.2fx\n", cpu_time_ms / gpu_time_ms);
    }

    // Cleanup
    cudaEventDestroy(gpuStart);
    cudaEventDestroy(gpuStop);
    cudaFree(d_input);
    cudaFree(d_output);
    free(h_input);
    free(h_output);

    return 0;
}

Writing reduction.cu


In [2]:
!nvcc reduction.cu -o reduction
!./reduction

GPU Sum: 1024.000000
CPU Sum: 1024.000000
Average GPU time per reduction: 0.028336 ms
Average CPU time per reduction: 0.003427 ms
Speedup (CPU/GPU): 0.12x
